In [ ]:
from langchain_community.document_loaders import JSONLoader
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.prompts import ChatPromptTemplate
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.chains import RetrievalQA
from langchain_ollama import ChatOllama

loader = JSONLoader(
    file_path="/Users/muhammadzuamaalamin/Documents/RisetTextMining/Qwen_json_20260313_krtrbraab.json",
    jq_schema=".[]",      # ambil setiap item dalam array
    text_content=False   # biar otomatis string
)

docs = loader.load()
print(f"Jumlah dokumen: {len(docs)}")
print(docs[0].page_content)


Jumlah dokumen: 71
{"id": "psl_1_ay_1", "pasal": 1, "ayat": 1, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Negara Indonesia ialah Negara kesatuan yang berbentuk Republik."}


In [3]:
for i in range(len(docs)):
    print(docs[i].page_content)

{"id": "psl_1_ay_1", "pasal": 1, "ayat": 1, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Negara Indonesia ialah Negara kesatuan yang berbentuk Republik."}
{"id": "psl_1_ay_2", "pasal": 1, "ayat": 2, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Kedaulatan adalah di tangan rakyat, dan dilakukan sepenuhnya oleh Majelis Permusyawaratan Rakyat."}
{"id": "psl_2_ay_1", "pasal": 2, "ayat": 1, "bab": "BAB II - MAJELIS PERMUSYAWARATAN RAKYAT", "content": "Majelis Permusyawaratan Rakyat terdiri atas anggota-anggota Dewan Perwakilan Rakyat, ditambah dengan utusan-utusan dari daerah-daerah dan golongan-golongan, menurut aturan yang ditetapkan dengan undang-undang."}
{"id": "psl_2_ay_2", "pasal": 2, "ayat": 2, "bab": "BAB II - MAJELIS PERMUSYAWARATAN RAKYAT", "content": "Majelis Permusyawaratan Rakyat bersidang sedikitnya sekali dalam lima tahun di ibukota negara."}
{"id": "psl_2_ay_3", "pasal": 2, "ayat": 3, "bab": "BAB II - MAJELIS PERMUSYAWARATAN RAKYAT", "content": "Segala putusan 

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

texts = text_splitter.split_documents(docs)
texts

[Document(metadata={'source': '/Users/muhammadzuamaalamin/Documents/RisetTextMining/Qwen_json_20260313_krtrbraab.json', 'seq_num': 1}, page_content='{"id": "psl_1_ay_1", "pasal": 1, "ayat": 1, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Negara Indonesia ialah Negara kesatuan yang berbentuk Republik."}'),
 Document(metadata={'source': '/Users/muhammadzuamaalamin/Documents/RisetTextMining/Qwen_json_20260313_krtrbraab.json', 'seq_num': 2}, page_content='{"id": "psl_1_ay_2", "pasal": 1, "ayat": 2, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Kedaulatan adalah di tangan rakyat, dan dilakukan sepenuhnya oleh Majelis Permusyawaratan Rakyat."}'),
 Document(metadata={'source': '/Users/muhammadzuamaalamin/Documents/RisetTextMining/Qwen_json_20260313_krtrbraab.json', 'seq_num': 3}, page_content='{"id": "psl_2_ay_1", "pasal": 2, "ayat": 1, "bab": "BAB II - MAJELIS PERMUSYAWARATAN RAKYAT", "content": "Majelis Permusyawaratan Rakyat terdiri atas anggota-anggota Dewan Perwakilan Rakyat

In [5]:
texts[0]

Document(metadata={'source': '/Users/muhammadzuamaalamin/Documents/RisetTextMining/Qwen_json_20260313_krtrbraab.json', 'seq_num': 1}, page_content='{"id": "psl_1_ay_1", "pasal": 1, "ayat": 1, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Negara Indonesia ialah Negara kesatuan yang berbentuk Republik."}')

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="/Users/muhammadzuamaalamin/Documents/fintunellm/model/bge-m3",
    model_kwargs={"device": "cpu"}
)

db_path = "faiss_index"

if not os.path.exists(db_path):
    vectorstore = FAISS.from_documents(texts, embeddings)
    vectorstore.save_local(db_path)
else:
    vectorstore = FAISS.load_local(
        db_path,
        embeddings,
        allow_dangerous_deserialization=True
    )


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [10]:
query = "bunyi pasal 1 uud 1945  ?"
results = vectorstore.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nResult {i} | score={score:.4f}")
    print(doc.page_content[:400])


Result 1 | score=0.9940
{"id": "psl_3_ay_1", "pasal": 3, "ayat": 1, "bab": "BAB II - MAJELIS PERMUSYAWARATAN RAKYAT", "content": "Majelis Permusyawaratan Rakyat menetapkan Undang-Undang Dasar dan garis-garis besar dari ada haluan negara."}

Result 2 | score=1.0006
{"id": "psl_1_ay_1", "pasal": 1, "ayat": 1, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Negara Indonesia ialah Negara kesatuan yang berbentuk Republik."}

Result 3 | score=1.0391
{"id": "psl_1_ay_2", "pasal": 1, "ayat": 2, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Kedaulatan adalah di tangan rakyat, dan dilakukan sepenuhnya oleh Majelis Permusyawaratan Rakyat."}

Result 4 | score=1.0414
{"id": "psl_37_ay_1", "pasal": 37, "ayat": 1, "bab": "BAB XVI - PERUBAHAN UNDANG-UNDANG DASAR", "content": "Untuk mengubah Undang-Undang Dasar sekurang-kurangnya 2/3 dari pada jumlah anggota Majelis Permusyawaratan Rakyat harus hadir."}

Result 5 | score=1.0549
{"id": "psl_ap_2_ay_1", "pasal": 2, "ayat": 1, "bab": "ATURAN PE

In [ ]:
bm25 = BM25Retriever.from_documents(texts)
bm25.k = 5

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.5, 0.5]
)

In [12]:
query = "Apa hak-hak warga negara yang dijamin dalam UUD 1945? ?"

docs = hybrid_retriever.get_relevant_documents(query)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nResult {i} | score={score:.4f}")
    print(doc.page_content[:400])



Result 1 | score=0.9940
{"id": "psl_3_ay_1", "pasal": 3, "ayat": 1, "bab": "BAB II - MAJELIS PERMUSYAWARATAN RAKYAT", "content": "Majelis Permusyawaratan Rakyat menetapkan Undang-Undang Dasar dan garis-garis besar dari ada haluan negara."}

Result 2 | score=1.0006
{"id": "psl_1_ay_1", "pasal": 1, "ayat": 1, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Negara Indonesia ialah Negara kesatuan yang berbentuk Republik."}

Result 3 | score=1.0391
{"id": "psl_1_ay_2", "pasal": 1, "ayat": 2, "bab": "BAB I - BENTUK DAN KEDAULATAN", "content": "Kedaulatan adalah di tangan rakyat, dan dilakukan sepenuhnya oleh Majelis Permusyawaratan Rakyat."}

Result 4 | score=1.0414
{"id": "psl_37_ay_1", "pasal": 37, "ayat": 1, "bab": "BAB XVI - PERUBAHAN UNDANG-UNDANG DASAR", "content": "Untuk mengubah Undang-Undang Dasar sekurang-kurangnya 2/3 dari pada jumlah anggota Majelis Permusyawaratan Rakyat harus hadir."}

Result 5 | score=1.0549
{"id": "psl_ap_2_ay_1", "pasal": 2, "ayat": 1, "bab": "ATURAN PE

In [ ]:
prompt_template = ChatPromptTemplate.from_template("""
Kamu adalah asisten ahli hukum tata negara Indonesia yang memahami Undang-Undang Dasar 1945.

Jawablah pertanyaan berikut HANYA berdasarkan konteks dokumen yang diberikan.
Berikan jawaban yang jelas, akurat, dan mudah dipahami.
Jika relevan, sebutkan pasal dan ayat yang menjadi dasar jawabanmu.

Pertanyaan:
{question}

Konteks:
{context}

Panduan menjawab:
- Jawab berdasarkan isi dokumen saja
- Sebutkan "Pasal X Ayat Y" jika mengutip pasal tertentu
- Jangan menambahkan informasi di luar konteks dokumen

Jika jawaban tidak ditemukan dalam dokumen, katakan:
"Maaf, informasi tersebut tidak dijelaskan secara spesifik dalam Undang-Undang Dasar 1945."
""")

In [16]:


llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)
response = qa_chain.invoke(
    {"query": "Apa hak-hak warga negara yang dijamin dalam UUD 1945?"}
)

print(response["result"])


KeyboardInterrupt: 

In [18]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,  # ← tinggal ganti llm di atas
    retriever=hybrid_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template},
    return_source_documents=True
)

response = qa_chain.invoke(
    {"query": "Apa hak-hak warga negara yang dijamin dalam UUD 1945?"}
)
print(response["result"])

KeyboardInterrupt: 

In [17]:
import importlib.metadata

packages = [
    "langchain",
    "langchain-community",
    "langchain-huggingface",
    "langchain-ollama",
    "langchain-text-splitters",
    "faiss-cpu",
]

for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"{pkg:35} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg:35} ❌ tidak terinstall")

langchain                           0.3.27
langchain-community                 0.3.27
langchain-huggingface               0.3.1
langchain-ollama                    0.3.6
langchain-text-splitters            0.3.9
faiss-cpu                           1.12.0
